# 💳 Part 3 — Technical Debt: Classify, Triage, Resolve
### Workshop 4 · LLMA4SE Summer School 2026

---

**⏱ Time:** ~50 min &nbsp;·&nbsp; **Needs:** T4 GPU runtime

Parts 1–2 operated on **code**. But most technical debt is first *reported in words* — issue trackers, PR comments, TODO notes. In this final part we:

1. **Classify** technical debt from issue-tracker text (the task behind **BEACon-TD**, Shivashankar et al., *JSS 2025*, which fine-tunes transformers to classify **13 debt types** from real issue trackers)
2. **Triage** it — because you can never fix everything, prioritisation *is* the skill
3. **Assemble the full pipeline**: issues + code in → verified fixes + a debt report out

```
 issue tracker ──► TD CLASSIFIER ──► TRIAGE AGENT ──► ranked backlog ─┐
                                                                      ▼
 codebase ──────► AUDITOR ──► REFACTORER ⇄ QA GATE ──► verified fix + 📄 DEBT REPORT
```

### 💡 The financial metaphor (worth internalising)

| Finance | Software |
|---|---|
| **Principal** | the cost of doing the proper fix |
| **Interest** | the *recurring* extra effort every sprint the debt stays (slower features, more bugs) |
| Taking a loan | shipping the quick hack to hit a deadline — *sometimes rational!* |
| Default 💥 | the rewrite-from-scratch / "nobody dares touch this module" state |

Debt isn't evil — **unmanaged, invisible** debt is. Managing starts with *naming and measuring* it.

## 3.0 · Setup (run me — ~3 min)

Standalone setup, as before.

In [ ]:
%pip install -q transformers accelerate radon pylint code-quality-analyzer pytest pandas
%pip install -q git+https://github.com/KarthikShivasankar/ml_smells_detector.git
print("✅ packages ready")

In [ ]:
with open('inventory.py', 'w') as f:
    f.write('''"""inventory.py -- Order processing for a small e-commerce shop.

This module works correctly (all tests pass!) but it is deliberately
full of code smells. Your agents will find and fix them.
"""

def helper_unused(x):          # SMELL: dead code -- never called anywhere
    return x * 2


class InventoryManager:
    """Manages stock and processes customer orders."""

    def __init__(self, items=[]):              # SMELL: mutable default argument
        self.items = {}
        for name, price, qty in items:
            self.items[name] = {"price": price, "qty": qty}
        self.log = []

    def add_item(self, name, price, qty, category, supplier, discount, taxable):
        # SMELL: long parameter list (7 params, most unused)
        self.items[name] = {"price": price, "qty": qty}
        return True

    def process_order(self, order):
        # SMELL: long method, deep nesting, magic numbers, duplication
        total = 0.0
        status = "ok"
        for name, qty in order:
            if name in self.items:
                if self.items[name]["qty"] >= qty:
                    if qty > 0:
                        price = self.items[name]["price"]
                        subtotal = price * qty
                        if subtotal > 100:                      # magic number
                            subtotal = subtotal - subtotal * 0.05   # magic number
                        if qty > 10:                            # magic number
                            subtotal = subtotal - subtotal * 0.02   # magic number
                        total = total + subtotal
                        self.items[name]["qty"] = self.items[name]["qty"] - qty
                        self.log.append("sold " + name)
                    else:
                        status = "invalid_qty"
                else:
                    status = "insufficient_stock"
            else:
                status = "unknown_item"
        total = total + total * 0.25            # magic number (VAT)
        return {"total": round(total, 2), "status": status}

    def refund_order(self, order):
        # SMELL: duplicated logic (mirror of process_order maths)
        total = 0.0
        for name, qty in order:
            if name in self.items:
                price = self.items[name]["price"]
                subtotal = price * qty
                if subtotal > 100:                              # magic number again
                    subtotal = subtotal - subtotal * 0.05
                if qty > 10:
                    subtotal = subtotal - subtotal * 0.02
                total = total + subtotal
                self.items[name]["qty"] = self.items[name]["qty"] + qty
        total = total + total * 0.25
        return {"total": round(total, 2), "status": "refunded"}

    def get_stock(self, name):
        if name in self.items:
            return self.items[name]["qty"]
        return 0
''')
with open('test_inventory.py', 'w') as f:
    f.write('''"""test_inventory.py -- Behaviour-preserving safety net.

These tests define the PUBLIC CONTRACT of the module. Any refactoring
your agents perform MUST keep every one of these green.
"""
import pytest
from inventory import InventoryManager


@pytest.fixture
def mgr():
    return InventoryManager([("widget", 10.0, 100), ("gizmo", 25.0, 5)])


def test_simple_order(mgr):
    result = mgr.process_order([("widget", 2)])
    assert result["status"] == "ok"
    assert result["total"] == 25.0          # 20 + 25% VAT


def test_bulk_discount_applied(mgr):
    # 20 widgets = 200 -> -5% (>100) -> -2% (>10 units) -> +25% VAT
    result = mgr.process_order([("widget", 20)])
    assert result["total"] == 232.75


def test_stock_is_decremented(mgr):
    mgr.process_order([("widget", 2)])
    assert mgr.get_stock("widget") == 98


def test_insufficient_stock(mgr):
    result = mgr.process_order([("gizmo", 99)])
    assert result["status"] == "insufficient_stock"


def test_unknown_item(mgr):
    result = mgr.process_order([("nonexistent", 1)])
    assert result["status"] == "unknown_item"


def test_refund_restores_stock(mgr):
    mgr.process_order([("widget", 2)])
    mgr.refund_order([("widget", 2)])
    assert mgr.get_stock("widget") == 100


def test_no_shared_state_between_instances():
    a = InventoryManager()
    b = InventoryManager()
    a.items["x"] = {"price": 1, "qty": 1}
    assert "x" not in b.items or a.items is not b.items
''')
print('✅ target project recreated')

In [ ]:
# Recreate the ML project from Part 1 (its smells go in the report too)
import os, subprocess, pathlib
os.makedirs('ml_project', exist_ok=True)
with open('ml_project/train_model.py', 'w') as f:
    f.write('''"""train_model.py -- Churn-prediction training script.

It trains fine... but is it reproducible? Is it healthy ML code?
Your ML Auditor agent (powered by MLScent) will tell you.
"""
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


def load_data(path):
    df = pd.read_csv(path)
    for i in range(len(df)):                      # pandas: unnecessary iteration
        if df["age"][i] == np.nan:                # numpy: NaN equality (always False!)
            df["age"][i] = 0                      # pandas: chain indexing
    return df


def train():
    df = load_data("churn.csv")
    X = df.drop("label", axis=1).values
    y = df["label"].values
    # sklearn: no feature scaling, no pipeline, no random_state
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33)

    model = nn.Sequential(nn.Linear(X.shape[1], 64), nn.ReLU(), nn.Linear(64, 2))
    opt = torch.optim.Adam(model.parameters(), lr=0.003)   # hardcoded hyperparams
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(100):                      # no early stopping, no checkpoints
        out = model(torch.tensor(X_train, dtype=torch.float32))
        loss = loss_fn(out, torch.tensor(y_train))
        loss.backward()                           # pytorch: missing opt.zero_grad()
        opt.step()

    preds = model(torch.tensor(X_test, dtype=torch.float32)).argmax(1).numpy()
    print("accuracy:", accuracy_score(y_test, preds))   # over-reliance on accuracy
    # no torch.manual_seed / np.random.seed anywhere -> unreproducible


if __name__ == "__main__":
    train()
''')

def run_mlscent(project_dir: str = "ml_project") -> str:
    """MLScent: 76 ML-specific anti-pattern detectors (Shivashankar, CAIN 2025).
    Findings land in output/analysis_report.txt, so we run the CLI then read it."""
    subprocess.run(["ml_smell_detector", "analyze", project_dir],
                   capture_output=True, text=True, timeout=300)
    report = pathlib.Path("output/analysis_report.txt")
    return report.read_text()[:3500] if report.exists() else "MLScent produced no report"
print('✅ ml_project recreated')

In [ ]:
# ============================================================
#  Load a small open-weights code LLM (fits free Colab T4 GPU)
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-1.5B-Instruct"   # ~3 GB in fp16
# Slower machine / CPU-only fallback:
# MODEL_NAME = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {MODEL_NAME} on {device} ...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
)
print("Model loaded ✔")


def llm(user_prompt: str, system_prompt: str = "You are a helpful assistant.",
        max_new_tokens: int = 1024, temperature: float = 0.2) -> str:
    """One call to our local LLM. Every agent in this workshop uses this."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=temperature > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    ).strip()


## 3.1 · A slice of a real-world issue tracker

Below: 10 issues of the kind BEACon-TD was trained on (paraphrased composites of real open-source issues). Each hides a **debt type** — sometimes several. We keep hand labels so we can score our classifier.

In [ ]:
ISSUES = [
 {"id": 101, "text": "The OrderService class is 3000 lines and does payment, shipping AND emails. Every change breaks something unrelated. We need to split it before adding new payment providers.", "gold": "design"},
 {"id": 102, "text": "There are zero tests for the refund flow. We only find regressions when customers complain. Adding tests keeps getting postponed for feature work.", "gold": "test"},
 {"id": 103, "text": "The README still describes the v1 API. New joiners lose days because the setup guide is wrong and the architecture diagram shows services we deleted last year.", "gold": "documentation"},
 {"id": 104, "text": "We're pinned to Django 2.2 which reached end-of-life. Security patches no longer land and two dependencies refuse to install alongside it.", "gold": "dependency"},
 {"id": 105, "text": "TODO left from the March crunch: error handling in the CSV importer just swallows exceptions with a bare except and returns None. Works until it doesn't.", "gold": "defect"},
 {"id": 106, "text": "The nightly ETL takes 6 hours because it re-reads the entire table every run. A watermark column was proposed in 2023 but never implemented.", "gold": "design"},
 {"id": 107, "text": "Deployment is a 14-step manual runbook involving three people and an SSH session. One typo in step 9 took prod down last month. We need CI/CD.", "gold": "build"},
 {"id": 108, "text": "Variable names in the pricing module are a, b, tmp2 and data_final_v3. Code review of any pricing change takes twice as long as it should.", "gold": "code"},
 {"id": 109, "text": "Our fork of the auth library diverged 200 commits from upstream. Merging upstream security fixes now takes a full sprint each time.", "gold": "dependency"},
 {"id": 110, "text": "The ML model in production was trained on 2022 data and nobody saved the training script or the random seed. Retraining reproducibly is currently impossible.", "gold": "code"},
]
print(f"{len(ISSUES)} issues loaded")

## 3.2 · Agent 4: the **TD Classifier**

BEACon-TD's production answer is a *fine-tuned transformer* — small, fast, cheap, consistent. Today we approximate the task with our instruct LLM + a **constrained-label prompt** (zero-shot classification). Note the constraint pattern: we give the model a *closed* label set and force a one-word answer — leaving no room to freestyle.

Label set (simplified from BEACon-TD's 13 types):
`design · code · test · documentation · dependency · build · defect · requirement`

In [ ]:
LABELS = ["design", "code", "test", "documentation",
          "dependency", "build", "defect", "requirement"]

CLASSIFIER_PROMPT = f"""You classify technical-debt reports from issue trackers.
Allowed labels (choose EXACTLY one): {", ".join(LABELS)}.

Definitions:
- design: architectural problems, god classes, wrong abstractions, inefficient designs
- code: poor readability/naming, code smells, missing reproducibility of code artifacts
- test: missing/weak/flaky tests, low coverage
- documentation: missing or outdated docs/diagrams/guides
- dependency: outdated/EOL libraries, diverged forks, version conflicts
- build: manual/fragile build, deployment or CI/CD problems
- defect: known bugs or error-handling gaps deliberately left in the code
- requirement: implementation diverged from what was actually required

Reply with ONLY the label word. Nothing else."""

def classify_issue(text: str) -> str:
    reply = llm(f"ISSUE:\n{text}\n\nLabel:", system_prompt=CLASSIFIER_PROMPT,
                max_new_tokens=8, temperature=0.0)
    reply = reply.lower().strip().split()[0].strip(".,:")
    return reply if reply in LABELS else "code"   # snap to label set

# Classify everything and score against gold labels
import pandas as pd
rows = []
for issue in ISSUES:
    pred = classify_issue(issue["text"])
    rows.append({"id": issue["id"], "gold": issue["gold"], "predicted": pred,
                 "✓": "✅" if pred == issue["gold"] else "❌",
                 "issue": issue["text"][:70] + "…"})
df = pd.DataFrame(rows)
accuracy = (df["gold"] == df["predicted"]).mean()
print(f"Zero-shot accuracy: {accuracy:.0%}\n")
df

### 🤔 Reading the errors

Look at the ❌ rows. Most misclassifications are **genuinely ambiguous** — issue 110 (unreproducible ML training) is arguably `code`, `design`, *and* a process problem. Real issues carry multiple debt types at once; BEACon-TD therefore treats this as **multi-label** classification, and its fine-tuned models beat zero-shot prompting on consistency and cost — the right tool once you have labelled data.

> 🧪 **Try it (3 min):** improve the accuracy *without touching the model* — edit only the label definitions in the prompt, or add one worked example per label ("few-shot"). How high can you push it? Which single change helped most?

## 3.3 · Agent 5: the **Triage Agent** — from list to strategy

Classification says *what kind*; triage says *what first*. We score each issue on the two axes of the debt quadrant:

- **Interest** 📈 — how much recurring pain per sprint? (1–5)
- **Principal** 💰 — how expensive is the proper fix? (1–5)

Priority rule of thumb: **high-interest / low-principal first** (quick wins), and *schedule* high-interest / high-principal items (they never fit "between tasks").

In [ ]:
TRIAGE_PROMPT = """You are an engineering manager triaging technical debt.
For the issue, estimate:
- interest: recurring pain per sprint if NOT fixed, integer 1 (minor) to 5 (severe)
- principal: effort of the proper fix, integer 1 (hours) to 5 (multi-sprint)
- rationale: one short sentence

Reply ONLY with a JSON object: {"interest": int, "principal": int, "rationale": str}"""

import re, json

def extract_json(text):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    return json.loads(m.group(0)) if m else {"interest": 3, "principal": 3, "rationale": "parse failed"}

def triage(issue) -> dict:
    reply = llm(f"ISSUE:\n{issue['text']}", system_prompt=TRIAGE_PROMPT,
                max_new_tokens=120, temperature=0.1)
    score = extract_json(reply)
    # priority: pain per unit of effort (add small epsilon vs division by zero)
    score["priority"] = round(score.get("interest", 3) / max(score.get("principal", 3), 1), 2)
    return score

backlog = []
for issue in ISSUES:
    s = triage(issue)
    backlog.append({"id": issue["id"], "type": classify_issue(issue["text"]),
                    "interest": s.get("interest"), "principal": s.get("principal"),
                    "priority": s["priority"], "rationale": s.get("rationale", "")[:60]})

pd.DataFrame(sorted(backlog, key=lambda r: -r["priority"]))

> 💬 **Discuss (2 min):** the LLM just made *resource-allocation judgements*. Would you let this ranking drive your sprint planning directly? What human checkpoint would you insert, and why there specifically? *(There's no consensus answer — the trade-off between automation speed and judgement accountability is an open research and management question.)*

## 3.4 · 🏗️ Capstone: the full pipeline

Everything, assembled. One function that takes a codebase + its issues and produces:
verified refactored code **and** a `TECH_DEBT_REPORT.md` a team could actually act on.

*(The cell below re-defines the Part 2 agents compactly so this notebook stands alone — skim it, you built every line of it already.)*

In [ ]:
# ── Part 2 team, compact edition ─────────────────────────────────
import subprocess, ast, os, shutil, tempfile, difflib
from dataclasses import dataclass, field

def run_radon(p):
    cc = subprocess.run(["radon","cc","-s",p],capture_output=True,text=True).stdout
    mi = subprocess.run(["radon","mi","-s",p],capture_output=True,text=True).stdout
    return f"COMPLEXITY:\n{cc}\nMAINTAINABILITY:\n{mi}"

def run_pylint(p):
    raw = subprocess.run(["pylint",p,"--output-format=json","--disable=C0114,C0115,C0116"],
                         capture_output=True,text=True).stdout
    try: return "\n".join(f"L{i['line']}: [{i['symbol']}] {i['message']}" for i in json.loads(raw)[:15]) or "clean"
    except json.JSONDecodeError: return raw[:1200]

def extract_json_arr(t):
    m = re.search(r"\[.*\]", t, re.DOTALL); return json.loads(m.group(0)) if m else []

def extract_code_block(t):
    b = re.findall(r"```(?:python)?\s*\n(.*?)```", t, re.DOTALL)
    if not b: raise ValueError("no code block")
    return b[-1].strip()+"\n"

AUDITOR_PROMPT = """You are a senior code reviewer. List the most important code smells.
Reply ONLY a JSON array of {"smell":str,"location":str,"severity":str,"why":str,"fix":str}. Max 6."""

REFACTORER_PROMPT = """You are an expert Python refactoring engineer. Rewrite the ENTIRE module
fixing the smells. HARD RULES: identical public API and behaviour (same returns, totals, statuses);
stdlib only; named constants for magic numbers; extract duplicated pricing logic; fix mutable
default; remove dead code. Output ONLY one fenced python code block."""

def audit(path):
    ev = f"radon:\n{run_radon(path)}\npylint:\n{run_pylint(path)}"
    return extract_json_arr(llm(f"FILE:\n```python\n{open(path).read()}\n```\nEVIDENCE:\n{ev}\nJSON now.",
                                system_prompt=AUDITOR_PROMPT, max_new_tokens=800, temperature=0.1))

def avg_cc(path):
    d = json.loads(subprocess.run(["radon","cc","-j",path],capture_output=True,text=True).stdout)
    s = [b["complexity"] for bl in d.values() for b in bl]
    return sum(s)/len(s) if s else 0.0

def qa_verify(original_path, candidate_code):
    try: ast.parse(candidate_code)
    except SyntaxError as e: return {"ok": False, "why": f"syntax: {e}"}
    with tempfile.TemporaryDirectory() as sb:
        open(os.path.join(sb,"inventory.py"),"w").write(candidate_code)
        shutil.copy("test_inventory.py", sb)
        r = subprocess.run(["python","-m","pytest","test_inventory.py","-q"],
                           cwd=sb,capture_output=True,text=True,timeout=120)
        if r.returncode != 0: return {"ok": False, "why": "tests failed: "+r.stdout[-300:]}
        before, after = avg_cc(original_path), avg_cc(os.path.join(sb,"inventory.py"))
    if after > before: return {"ok": False, "why": f"complexity worsened {before:.2f}→{after:.2f}"}
    return {"ok": True, "cc_before": round(before,2), "cc_after": round(after,2)}

print("✅ team re-assembled")

In [ ]:
def full_pipeline(code_path: str, issues: list, max_iterations: int = 3) -> str:
    """Issues + code in → verified fix + Markdown debt report out."""
    log = lambda *a: print("  ", *a)

    print("① Classifying & triaging the issue backlog …")
    backlog = []
    for it in issues:
        s = triage(it)
        backlog.append({**it, "type": classify_issue(it["text"]), **s})
    backlog.sort(key=lambda r: -r["priority"])

    print("② Auditing the code …")
    findings = audit(code_path)
    log(f"{len(findings)} findings:", ", ".join(f.get("smell","?") for f in findings)[:80])

    print("③ Refactor ⇄ QA loop …")
    original, accepted, verdict, feedback = open(code_path).read(), None, {}, ""
    for i in range(1, max_iterations+1):
        extra = f"\nPREVIOUS ATTEMPT FAILED: {feedback}" if feedback else ""
        reply = llm(f"MODULE:\n```python\n{original}\n```\nFINDINGS:\n{json.dumps(findings)}{extra}\nRewrite now.",
                    system_prompt=REFACTORER_PROMPT, max_new_tokens=1600, temperature=0.1)
        try: candidate = extract_code_block(reply)
        except ValueError as e: feedback = str(e); log(f"iter {i}: ❌ {e}"); continue
        verdict = qa_verify(code_path, candidate)
        log(f"iter {i}:", "✅ accepted" if verdict["ok"] else f"❌ {verdict['why'][:70]}")
        if verdict["ok"]: accepted = candidate; break
        feedback = verdict["why"][:300]

    print("④ Writing TECH_DEBT_REPORT.md …")
    top = backlog[:5]
    lines = ["# Technical Debt Report", "",
             f"*Generated by a cooperative LLM-agent pipeline · model: {MODEL_NAME}*", "",
             "## 1 · Prioritised issue backlog (top 5)", "",
             "| Rank | Issue | Type | Interest | Principal | Priority |", "|--|--|--|--|--|--|"]
    for rank, r in enumerate(top, 1):
        lines.append(f"| {rank} | #{r['id']} {r['text'][:55]}… | {r['type']} | "
                     f"{r['interest']} | {r['principal']} | {r['priority']} |")
    lines += ["", "## 2 · Code audit findings", ""]
    for f in findings:
        lines.append(f"- **{f.get('smell','?')}** ({f.get('severity','?')}, {f.get('location','?')}): "
                     f"{f.get('why','')} → *{f.get('fix','')}*")
    lines += ["", "## 3 · Automated refactoring outcome", ""]
    if accepted:
        open("inventory_refactored.py","w").write(accepted)
        lines += [f"- ✅ Patch **accepted**: all 7 behaviour tests pass; "
                  f"avg complexity {verdict['cc_before']} → {verdict['cc_after']}",
                  "- Verified code saved to `inventory_refactored.py`"]
    else:
        lines += [f"- ❌ No patch met the QA bar within {max_iterations} iterations "
                  f"(last reason: {verdict.get('why','n/a')[:120]}). Original code kept — "
                  "**the gate held; nothing unverified shipped.**"]
    print("⑤ Scanning ML code with MLScent …")
    ml_report = run_mlscent("ml_project")
    counts = ml_report.split("Smell Counts:")[-1].strip().splitlines()[:8]
    lines += ["", "## 4 · ML-specific smells (`ml_project/` · MLScent)", ""]
    lines += [f"- {c.strip()}" for c in counts if c.strip()]
    lines += ["", "*Full MLScent report: `output/analysis_report.txt` — each finding includes a How-to-fix.*"]

    report = "\n".join(lines)
    open("TECH_DEBT_REPORT.md","w").write(report)
    return report

report = full_pipeline("inventory.py", ISSUES, max_iterations=3)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(report))

🏆 **You built an end-to-end technical-debt management pipeline** — perception (tools), reasoning (LLM), action (refactoring), verification (QA gates), and reporting — on a free GPU, with a 1.5-billion-parameter model. The architecture is what mattered, not the model size.

## 3.5 · ✍️ Final challenge (rest of the session)

**Feed the pipeline your own code.** Paste any Python module of yours (or grab one from a repo) into the cell below, *write 2–3 behaviour tests for it*, and run `full_pipeline` on it.

- No tests you trust? → notice how uncomfortable that feels. **That discomfort is test debt, quantified emotionally.**
- Model mangles your code? → the QA gate rejects it. Working as designed.
- Everything passes first try? → raise the bar in `qa_verify` (make MI improvement mandatory) and see what happens.

In [ ]:
%%writefile my_module.py
# 🖊️ Paste YOUR code here, then adapt test_inventory.py-style tests for it,
# point qa_verify's sandbox at your test file, and run:
#   report = full_pipeline("my_module.py", ISSUES)

def example(a, b):
    return a + b

## 3.6 · Where to go from here 🧭

**Scaling this up in the real world**

| Today's toy | Production equivalent |
|---|---|
| `WorkflowState` dataclass | LangGraph state graphs · AutoGen conversations · CrewAI crews |
| `llm()` on a 1.5B model | API frontier models, or fine-tuned small models (cheaper + more consistent for narrow tasks) |
| zero-shot TD classifier | **BEACon-TD / TD-Suite** fine-tuned transformers (13 debt types) |
| 3 hand-rolled tools | **PyExamine** (49 metrics) · **MLScent** (76 ML-specific anti-patterns for TensorFlow/PyTorch) · full linter farms |
| 7 pytest tests as the gate | full CI: coverage thresholds, mutation testing, canary deploys |

**Papers & tools from today** *(all by today's instructor & colleagues — ask questions now, this is the rare chance!)*
- PyExamine — *MSR 2025* · `pip install code-quality-analyzer` · [github.com/KarthikShivasankar/python_smells_detector](https://github.com/KarthikShivasankar/python_smells_detector)
- MLScent — *CAIN 2025* · [github.com/KarthikShivasankar/ml_smells_detector](https://github.com/KarthikShivasankar/ml_smells_detector)
- BEACon-TD — *Journal of Systems and Software, 2025* · TD-Suite: [github.com/KarthikShivasankar/text_classification](https://github.com/KarthikShivasankar/text_classification)
- *Enhancing Python Code Maintainability through LLM-Based Approaches* — Shivashankar & Martini, 2025 (the research version of today's Part 2)
- Fowler, *Refactoring* (2nd ed.) — the smell taxonomy everything builds on
- Cunningham (1992) — the original "debt" metaphor, 2 pages, worth reading verbatim

## 3.7 · Final checkpoint ✅

1. Why does a fine-tuned small classifier often beat a prompted large LLM for TD classification in production?
2. Explain "interest vs principal" to an imaginary product manager in two sentences.
3. What is the single most important component to add before letting this pipeline near a real repository?

<details><summary>Answers</summary>

1. Consistency, latency, cost, and privacy: a fine-tuned model gives stable labels at ~1000× lower cost per issue, can run on-prem, and doesn't drift with prompt wording.
2. "The principal is what the proper fix costs once; the interest is what the shortcut costs us *every sprint until then*. We're not paying down debt for cleanliness — we're cancelling a recurring payment."
3. Human review before any write action (plus CI as a second gate): agents propose, verified pipelines + humans dispose.
</details>

---
*Thanks for building with us! — Karthik & Adela · LLMA4SE 2026, Day 3*